# Battery Shortage Predictor (E-Scooter Swapping Station)
Run **Runtime > Run all**. At the end, download `report_bundle.zip` and upload it to Claude to get the PDF report.

Optional: upload `battery_data_v1.csv` and `battery_data_v2.csv` using the file panel (left) into `/content/` before running Cell 2. If you skip this, the notebook generates the identical datasets automatically (fixed random seed).

## Cell 1: Install and project setup

In [2]:
!pip -q install mlflow scikit-learn pandas numpy fastapi uvicorn httpx pytest reportlab joblib requests matplotlib

import os, shutil
os.makedirs("/content/battery_project", exist_ok=True)
%cd /content/battery_project
for d in ["src", "tests", "data/raw", "data/feature_store", "models", "reports", ".github/workflows"]:
    os.makedirs(d, exist_ok=True)
open("src/__init__.py", "w").close()
open("tests/__init__.py", "w").close()

# ---- Manual dataset upload ----
from google.colab import files
print("Select battery_data_v1.csv and battery_data_v2.csv (hold Ctrl/Cmd to pick both):")
uploaded = files.upload()

for name in uploaded:
    low = name.lower()
    if "v1" in low:
        shutil.move(name, "data/raw/battery_data_v1.csv")
    elif "v2" in low:
        shutil.move(name, "data/raw/battery_data_v2.csv")
    else:
        print("Skipped (name must contain v1 or v2):", name)

print("Files in data/raw:", os.listdir("data/raw"))

/content/battery_project
Select battery_data_v1.csv and battery_data_v2.csv (hold Ctrl/Cmd to pick both):


Saving battery_data_v1.csv to battery_data_v1.csv
Saving battery_data_v2.csv to battery_data_v2.csv
Files in data/raw: ['battery_data_v2.csv', 'battery_data_v1.csv']


## Cell 2: Dataset generator (Task 1: v1 normal, v2 college festival)

In [3]:
%%writefile src/data_gen.py
import numpy as np
import pandas as pd


def generate_dataset(n=250, version=1, seed=42):
    """version 1 = normal evenings, version 2 = college festival (higher demand)."""
    rng = np.random.default_rng(seed + version)
    festival = version == 2

    charged = np.clip(rng.normal(34, 6, n).round(), 10, 55).astype(int)
    finishing = rng.integers(0, 16, n)
    bookings = rng.poisson(12 if festival else 11, n)
    walkin = np.clip(rng.normal(26 if festival else 24, 6, n).round(), 5, None).astype(int)
    rain = np.clip(rng.exponential(5, n), 0, 40).round(1)
    rain[rng.random(n) < 0.5] = 0.0
    weekend = rng.integers(0, 2, n)
    event = (rng.random(n) < (0.4 if festival else 0.15)).astype(int)

    # demand multiplier: rain, weekend, nearby event, festival boost
    mult = 1 + 0.015 * rain + 0.15 * weekend + 0.30 * event + (0.05 if festival else 0.0)

    # simulated customer arrivals
    arrivals = rng.binomial(bookings, 0.9) + rng.poisson(walkin * mult)
    # simulated actual supply at 7 PM (90% of charging batteries finish on time)
    supply = charged + rng.binomial(finishing, 0.9)

    df = pd.DataFrame({
        "charged_batteries_5pm": charged,
        "batteries_finishing_by_7pm": finishing,
        "advance_bookings": bookings,
        "historical_walkin_demand": walkin,
        "rainfall_forecast_mm": rain,
        "is_weekend": weekend,
        "nearby_event": event,
        "simulated_arrivals": arrivals,
    })
    df["battery_shortage"] = (arrivals > supply).astype(int)
    df["dataset_version"] = version
    return df


if __name__ == "__main__":
    for v in (1, 2):
        d = generate_dataset(version=v)
        print(f"v{v}: rows={len(d)}, shortage rate={d.battery_shortage.mean():.2%}")

Writing src/data_gen.py


## Cell 3: Preprocessing and validation (Task 2)

In [4]:
%%writefile src/preprocess.py
import numpy as np
import pandas as pd

RAW_FEATURES = [
    "charged_batteries_5pm", "batteries_finishing_by_7pm", "advance_bookings",
    "historical_walkin_demand", "rainfall_forecast_mm", "is_weekend", "nearby_event",
]
ENGINEERED = ["expected_battery_supply", "booking_pressure_ratio"]
FEATURES = RAW_FEATURES + ENGINEERED
NON_NEGATIVE = RAW_FEATURES[:5]
TARGET = "battery_shortage"


def validate(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in RAW_FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df = df.copy()
    df[RAW_FEATURES] = df[RAW_FEATURES].fillna(df[RAW_FEATURES].median())
    for c in NON_NEGATIVE:
        if (df[c] < 0).any():
            raise ValueError(f"Invalid negative values found in column '{c}'")
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["expected_battery_supply"] = df["charged_batteries_5pm"] + df["batteries_finishing_by_7pm"]
    df["booking_pressure_ratio"] = df["advance_bookings"] / np.maximum(df["expected_battery_supply"], 1)
    return df


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """validate -> impute -> engineer. Used by training AND serving."""
    return engineer_features(validate(df))

Writing src/preprocess.py


## Cell 4: Feature store (Task 2)

In [5]:
%%writefile src/feature_store.py
import json
import os
from datetime import datetime
import pandas as pd

STORE = "data/feature_store"


class FeatureStore:
    def __init__(self, path=STORE):
        self.path = path
        os.makedirs(path, exist_ok=True)
        self.meta_file = os.path.join(path, "registry.json")

    def _meta(self):
        if os.path.exists(self.meta_file):
            with open(self.meta_file) as f:
                return json.load(f)
        return {}

    def _save(self, meta):
        with open(self.meta_file, "w") as f:
            json.dump(meta, f, indent=2)

    def write(self, name, version, df):
        key = f"{name}_v{version}"
        df.to_csv(os.path.join(self.path, key + ".csv"), index=False)
        meta = self._meta()
        meta[key] = {"rows": len(df), "columns": list(df.columns),
                     "created": datetime.now().isoformat(timespec="seconds"), "reads": 0}
        self._save(meta)

    def read(self, name, version):
        key = f"{name}_v{version}"
        df = pd.read_csv(os.path.join(self.path, key + ".csv"))
        meta = self._meta()
        meta[key]["reads"] += 1          # tracks feature reuse
        self._save(meta)
        return df

Writing src/feature_store.py


## Cell 5: Automated pipeline (Task 2)

In [6]:
%%writefile src/pipeline.py
import os
import pandas as pd
from src.data_gen import generate_dataset
from src.preprocess import preprocess
from src.feature_store import FeatureStore


def run_pipeline(versions=(1, 2)):
    fs = FeatureStore()
    os.makedirs("data/raw", exist_ok=True)
    for v in versions:
        path = f"data/raw/battery_data_v{v}.csv"
        if os.path.exists(path):                      # use the provided dataset if present
            raw = pd.read_csv(path)
            src = "loaded provided CSV"
        else:
            raw = generate_dataset(n=250, version=v)
            raw.to_csv(path, index=False)
            src = "generated"
        feats = preprocess(raw)
        fs.write("battery_features", v, feats)        # feature store
        print(f"[pipeline] v{v}: raw {src}, {len(feats)} engineered rows stored in feature store")


if __name__ == "__main__":
    run_pipeline()

Writing src/pipeline.py


## Cell 6: Training, comparison, MLflow registry (Task 3)

In [7]:
%%writefile src/train.py
import json
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.pipeline import run_pipeline
from src.feature_store import FeatureStore
from src.preprocess import FEATURES, TARGET

MODEL_NAME = "battery_shortage_model"
TRUSTED = ["sklearn.tree._tree.Tree"]   # we trained this model ourselves, so it is safe


def log_and_register(model):
    """Works across MLflow versions (2.x and 3.x)."""
    attempts = [
        dict(name="model", registered_model_name=MODEL_NAME, skops_trusted_types=TRUSTED),
        dict(artifact_path="model", registered_model_name=MODEL_NAME, skops_trusted_types=TRUSTED),
        dict(name="model", registered_model_name=MODEL_NAME),
        dict(artifact_path="model", registered_model_name=MODEL_NAME),
    ]
    last = None
    for kw in attempts:
        try:
            mlflow.sklearn.log_model(model, **kw)
            return
        except TypeError as e:
            last = e
    raise last


def main():
    run_pipeline()
    fs = FeatureStore()
    # Feature reuse: training reads engineered features from the store (no recomputation)
    df = pd.concat([fs.read("battery_features", 1), fs.read("battery_features", 2)], ignore_index=True)
    print(f"[train] loaded {len(df)} rows from feature store (v1 + v2)")

    X, y = df[FEATURES], df[TARGET]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    models = {
        "LogisticRegression": Pipeline([("scaler", StandardScaler()),
                                        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))]),
        "GradientBoosting": Pipeline([("scaler", StandardScaler()),
                                      ("clf", GradientBoostingClassifier(random_state=42))]),
    }

    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("battery_shortage")

    results, cms = {}, {}
    for name, pipe in models.items():
        with mlflow.start_run(run_name=name):
            pipe.fit(X_tr, y_tr)
            pred = pipe.predict(X_te)
            proba = pipe.predict_proba(X_te)[:, 1]
            m = {
                "accuracy": float(accuracy_score(y_te, pred)),
                "precision": float(precision_score(y_te, pred)),
                "recall": float(recall_score(y_te, pred)),
                "f1": float(f1_score(y_te, pred)),
                "roc_auc": float(roc_auc_score(y_te, proba)),
            }
            mlflow.log_param("model", name)
            mlflow.log_param("features", ",".join(FEATURES))
            mlflow.log_metrics(m)
            results[name] = m
            cms[name] = confusion_matrix(y_te, pred).tolist()
            print(name, {k: round(v, 3) for k, v in m.items()})
            print("confusion matrix:", cms[name])

    # best model: highest recall on shortage class (F1 as tie-breaker)
    best = max(results, key=lambda k: (round(results[k]["recall"], 4), results[k]["f1"]))
    print(f"\n[train] best model by recall: {best}")

    # save for FastAPI first, so serving never depends on MLflow
    joblib.dump(models[best], "models/best_model.joblib")
    with open("reports/results.json", "w") as f:
        json.dump({"best": best, "results": results, "confusion_matrices": cms,
                   "train_rows": len(X_tr), "test_rows": len(X_te)}, f, indent=2)

    with mlflow.start_run(run_name=f"register_{best}"):
        mlflow.log_param("model", best)
        mlflow.log_metrics(results[best])
        log_and_register(models[best])

    client = MlflowClient()
    latest = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
    client.set_registered_model_alias(MODEL_NAME, "champion", str(latest))
    print(f"[mlflow] registered '{MODEL_NAME}' version {latest} with alias 'champion'")


if __name__ == "__main__":
    main()

Writing src/train.py


## Cell 7: FastAPI service (Task 4)

In [8]:
%%writefile src/api.py
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

from src.preprocess import preprocess, FEATURES

app = FastAPI(title="Battery Shortage Predictor")
_model = None


def get_model():
    global _model
    if _model is None:
        _model = joblib.load("models/best_model.joblib")
    return _model


class StationConditions(BaseModel):
    charged_batteries_5pm: int = Field(ge=0)
    batteries_finishing_by_7pm: int = Field(ge=0)
    advance_bookings: int = Field(ge=0)
    historical_walkin_demand: int = Field(ge=0)
    rainfall_forecast_mm: float = Field(ge=0)
    is_weekend: int = Field(ge=0, le=1)
    nearby_event: int = Field(ge=0, le=1)


@app.get("/")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(c: StationConditions):
    df = preprocess(pd.DataFrame([c.model_dump()]))
    X = df[FEATURES]
    model = get_model()
    pred = int(model.predict(X)[0])
    prob = float(model.predict_proba(X)[0, 1])
    return {
        "prediction": "Shortage Expected" if pred == 1 else "Sufficient Batteries",
        "shortage_probability": round(prob, 3),
    }

Writing src/api.py


## Cell 8: Tests incl. negative battery test (Task 4)

In [9]:
%%writefile tests/test_pipeline.py
import pandas as pd
import pytest
from fastapi.testclient import TestClient

from src.data_gen import generate_dataset
from src.preprocess import preprocess
from src.api import app

client = TestClient(app)

GOOD = {"charged_batteries_5pm": 20, "batteries_finishing_by_7pm": 5, "advance_bookings": 15,
        "historical_walkin_demand": 30, "rainfall_forecast_mm": 5.0, "is_weekend": 1, "nearby_event": 1}


def test_dataset_size_and_target():
    df = generate_dataset(n=250, version=1)
    assert len(df) == 250 and set(df.battery_shortage.unique()) <= {0, 1}


def test_festival_version_has_more_shortages():
    assert generate_dataset(version=2).battery_shortage.mean() > generate_dataset(version=1).battery_shortage.mean()


def test_negative_battery_count_raises_error():
    df = generate_dataset(n=10)
    df.loc[0, "charged_batteries_5pm"] = -3
    with pytest.raises(ValueError):
        preprocess(df)


def test_engineered_features():
    out = preprocess(pd.DataFrame([GOOD]))
    assert out["expected_battery_supply"].iloc[0] == 25
    assert out["booking_pressure_ratio"].iloc[0] == pytest.approx(15 / 25)


def test_api_rejects_negative_batteries():
    bad = dict(GOOD, charged_batteries_5pm=-5)
    assert client.post("/predict", json=bad).status_code == 422


def test_api_prediction_label():
    r = client.post("/predict", json=GOOD)
    assert r.status_code == 200
    assert r.json()["prediction"] in ("Shortage Expected", "Sufficient Batteries")

Writing tests/test_pipeline.py


## Cell 9: Requirements

In [10]:
%%writefile requirements.txt
pandas
numpy
scikit-learn
mlflow
fastapi
uvicorn
httpx
pytest
joblib
reportlab
matplotlib

Writing requirements.txt


## Cell 10: GitHub Actions workflow (Task 4)

In [11]:
%%writefile .github/workflows/ci.yml
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - name: Build features and train model
        run: python -m src.train
      - name: Run tests
        run: pytest -v

Writing .github/workflows/ci.yml


In [12]:
%%writefile .gitignore
__pycache__/
mlruns/
mlflow.db
*.pyc
.pytest_cache/

Writing .gitignore


## Cell 11: README

In [13]:
%%writefile README.md
# Battery Shortage Predictor

Predicts whether an e-scooter battery-swapping station will run short of charged batteries between 5 PM and 7 PM.

- `src/data_gen.py` synthetic datasets (v1 normal, v2 college festival)
- `src/preprocess.py` validation and feature engineering
- `src/feature_store.py` and `src/pipeline.py` feature store and automated pipeline
- `src/train.py` Logistic Regression vs Gradient Boosting, MLflow registry
- `src/api.py` FastAPI service (`POST /predict`)
- `tests/` pytest suite, run by `.github/workflows/ci.yml`

Run: `pip install -r requirements.txt && python -m src.train && pytest -v && uvicorn src.api:app`

Writing README.md


## Cell 12: Run pipeline (creates dataset v1, v2 and the feature store)

In [14]:
!python -m src.data_gen
!python -m src.pipeline
!ls -la data/raw data/feature_store

v1: rows=250, shortage rate=36.80%
v2: rows=250, shortage rate=58.80%
[pipeline] v1: raw loaded provided CSV, 250 engineered rows stored in feature store
[pipeline] v2: raw loaded provided CSV, 250 engineered rows stored in feature store
data/feature_store:
total 36
drwxr-xr-x 2 root root  4096 Nov 18 09:01 .
drwxr-xr-x 4 root root  4096 Nov 18 08:59 ..
-rw-r--r-- 1 root root 11879 Nov 18 09:01 battery_features_v1.csv
-rw-r--r-- 1 root root 11899 Nov 18 09:01 battery_features_v2.csv
-rw-r--r-- 1 root root   938 Nov 18 09:01 registry.json

data/raw:
total 24
drwxr-xr-x 2 root root 4096 Nov 18 09:00 .
drwxr-xr-x 4 root root 4096 Nov 18 08:59 ..
-rw-r--r-- 1 root root 6699 Nov 18 09:00 battery_data_v1.csv
-rw-r--r-- 1 root root 6739 Nov 18 09:00 battery_data_v2.csv


## Cell 13: Train, compare and register in MLflow

In [15]:
!python -m src.train

[pipeline] v1: raw loaded provided CSV, 250 engineered rows stored in feature store
[pipeline] v2: raw loaded provided CSV, 250 engineered rows stored in feature store
[train] loaded 500 rows from feature store (v1 + v2)
2026/11/18 09:01:39 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/18 09:01:39 INFO mlflow.store.db.utils: Updating database tables
2026/11/18 09:01:43 INFO mlflow.tracking.fluent: Experiment with name 'battery_shortage' does not exist. Creating a new experiment.
LogisticRegression {'accuracy': 0.81, 'precision': 0.784, 'recall': 0.833, 'f1': 0.808, 'roc_auc': 0.895}
confusion matrix: [[41, 11], [8, 40]]
GradientBoosting {'accuracy': 0.84, 'precision': 0.82, 'recall': 0.854, 'f1': 0.837, 'roc_auc': 0.879}
confusion matrix: [[43, 9], [7, 41]]

[train] best model by recall: GradientBoosting
Successfully registered model 'battery_shortage_model'.
Created version '1' of model 'battery_shortage_model'.
[mlflow] registered 'battery_shortage_mo

## Cell 14: Run tests (saves output for the report)

In [16]:
!python -m pytest -v 2>&1 | tee reports/pytest_output.txt

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/battery_project
plugins: platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0
collecting ... collected 6 items

tests/test_pipeline.py::test_dataset_size_and_target PASSED              [ 16%]
tests/test_pipeline.py::test_festival_version_has_more_shortages PASSED  [ 33%]
tests/test_pipeline.py::test_negative_battery_count_raises_error PASSED  [ 50%]
tests/test_pipeline.py::test_engineered_features PASSED                  [ 66%]
tests/test_pipeline.py::test_api_rejects_negative_batteries PASSED       [ 83%]
tests/test_pipeline.py::test_api_prediction_label PASSED                 [100%]

============================== 6 passed in 2.40s ===============================


## Cell 15: Feature reuse evidence (`reads` counts how often training consumed stored features)

In [17]:
!cat data/feature_store/registry.json

{
  "battery_features_v1": {
    "rows": 250,
    "columns": [
      "charged_batteries_5pm",
      "batteries_finishing_by_7pm",
      "advance_bookings",
      "historical_walkin_demand",
      "rainfall_forecast_mm",
      "is_weekend",
      "nearby_event",
      "simulated_arrivals",
      "battery_shortage",
      "dataset_version",
      "expected_battery_supply",
      "booking_pressure_ratio"
    ],
    "created": "2026-11-18T09:01:36",
    "reads": 1
  },
  "battery_features_v2": {
    "rows": 250,
    "columns": [
      "charged_batteries_5pm",
      "batteries_finishing_by_7pm",
      "advance_bookings",
      "historical_walkin_demand",
      "rainfall_forecast_mm",
      "is_weekend",
      "nearby_event",
      "simulated_arrivals",
      "battery_shortage",
      "dataset_version",
      "expected_battery_supply",
      "booking_pressure_ratio"
    ],
    "created": "2026-11-18T09:01:36",
    "reads": 1
  }
}

## Cell 16: MLflow registry table and dataset summary

In [18]:
import json, pandas as pd, mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()

rows = []
for mv in client.search_model_versions("name='battery_shortage_model'"):
    aliases = list(getattr(mv, "aliases", []) or [])
    try:
        if client.get_model_version_by_alias("battery_shortage_model", "champion").version == mv.version:
            aliases = sorted(set(aliases + ["champion"]))
    except Exception:
        pass
    rows.append({"model": mv.name, "version": mv.version, "aliases": ",".join(aliases), "status": mv.status})
reg = pd.DataFrame(rows); reg.to_csv("reports/mlflow_registry.csv", index=False)
display(reg)

runs = mlflow.search_runs(experiment_names=["battery_shortage"])
cols = [c for c in runs.columns if c.startswith("metrics.") or c in ("tags.mlflow.runName", "params.model")]
runs[cols].to_csv("reports/mlflow_runs.csv", index=False)
display(runs[cols])

v1 = pd.read_csv("data/raw/battery_data_v1.csv"); v2 = pd.read_csv("data/raw/battery_data_v2.csv")
summary = {}
for name, d in (("v1", v1), ("v2", v2)):
    summary[name] = {"rows": int(len(d)),
        "avg_advance_bookings": round(float(d.advance_bookings.mean()), 2),
        "avg_walkin_demand": round(float(d.historical_walkin_demand.mean()), 2),
        "avg_charged_5pm": round(float(d.charged_batteries_5pm.mean()), 2),
        "event_rate": round(float(d.nearby_event.mean()), 3),
        "avg_rainfall_mm": round(float(d.rainfall_forecast_mm.mean()), 2),
        "avg_simulated_arrivals": round(float(d.simulated_arrivals.mean()), 2),
        "shortage_rate": round(float(d.battery_shortage.mean()), 3)}
json.dump(summary, open("reports/dataset_summary.json", "w"), indent=2)
display(pd.DataFrame(summary))

,model,version,aliases,status
0,battery_shortage_model,1,champion,READY


,metrics.f1,metrics.precision,metrics.roc_auc,metrics.recall,metrics.accuracy,params.model,tags.mlflow.runName
0,0.836735,0.820000,0.879006,0.854167,0.84,GradientBoosting,register_GradientBoosting
1,0.836735,0.820000,0.879006,0.854167,0.84,GradientBoosting,GradientBoosting
2,0.808081,0.784314,0.894631,0.833333,0.81,LogisticRegression,LogisticRegression


,v1,v2
rows,250.000,250.000
avg_advance_bookings,10.560,11.790
avg_walkin_demand,24.220,25.730
avg_charged_5pm,33.660,34.490
event_rate,0.128,0.368
avg_rainfall_mm,2.210,2.840
avg_simulated_arrivals,37.160,44.360
shortage_rate,0.368,0.588


## Cell 17: Charts for the report

In [19]:
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt2
res = json.load(open("reports/results.json")); summ = json.load(open("reports/dataset_summary.json"))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["v1 normal", "v2 festival"], [summ["v1"]["shortage_rate"]*100, summ["v2"]["shortage_rate"]*100], color=["#4c78a8", "#e45756"])
ax.set_ylabel("Shortage rate (%)"); ax.set_title("Shortage rate by dataset version")
for i, v in enumerate([summ["v1"]["shortage_rate"]*100, summ["v2"]["shortage_rate"]*100]): ax.text(i, v+1, f"{v:.1f}%", ha="center")
plt.tight_layout(); plt.savefig("reports/chart_shortage_rate.png", dpi=150); plt.show()

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
fig, ax = plt.subplots(figsize=(7, 3.5)); w = 0.38
for i, (name, m) in enumerate(res["results"].items()):
    ax.bar([x + i*w for x in range(len(metrics))], [m[k] for k in metrics], w, label=name)
ax.set_xticks([x + w/2 for x in range(len(metrics))]); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05); ax.legend(); ax.set_title("Model comparison (test set)")
plt.tight_layout(); plt.savefig("reports/chart_model_comparison.png", dpi=150); plt.show()

## Cell 18: Serve the model with FastAPI and get predictions

In [20]:
import subprocess, time, requests, json
server = subprocess.Popen(["uvicorn", "src.api:app", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/").status_code == 200: break
    except Exception:
        time.sleep(1)

high = {"charged_batteries_5pm": 15, "batteries_finishing_by_7pm": 4, "advance_bookings": 25,
        "historical_walkin_demand": 35, "rainfall_forecast_mm": 12.0, "is_weekend": 1, "nearby_event": 1}
low = {"charged_batteries_5pm": 45, "batteries_finishing_by_7pm": 12, "advance_bookings": 6,
       "historical_walkin_demand": 15, "rainfall_forecast_mm": 0.0, "is_weekend": 0, "nearby_event": 0}
bad = dict(low, charged_batteries_5pm=-4)

preds = []
for label, payload in [("High demand (festival evening, rain)", high), ("Low demand (quiet weekday)", low), ("Invalid negative battery count", bad)]:
    r = requests.post("http://127.0.0.1:8000/predict", json=payload)
    out = r.json() if r.status_code == 200 else "rejected: validation error"
    preds.append({"scenario": label, "input": payload, "status_code": r.status_code, "output": out})
    print(label, "->", r.status_code, out)
json.dump(preds, open("reports/predictions.json", "w"), indent=2)
server.terminate()

High demand (festival evening, rain) -> 200 {'prediction': 'Shortage Expected', 'shortage_probability': 0.999}
Low demand (quiet weekday) -> 200 {'prediction': 'Sufficient Batteries', 'shortage_probability': 0.004}
Invalid negative battery count -> 422 rejected: validation error


## Cell 19: Bundle everything for the report (download and upload to Claude)

In [21]:
import shutil
from google.colab import files
shutil.make_archive("/content/report_bundle", "zip", "/content/battery_project",
                    "reports")
shutil.make_archive("/content/project_code", "zip", "/content/battery_project")
files.download("/content/report_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 20 (optional): Push to GitHub
Create an empty repo on github.com and a Personal Access Token (scope `repo`). Edit the three values, then run.

In [22]:
USERNAME = "YOUR_GITHUB_USERNAME"
TOKEN = "YOUR_TOKEN"
REPO = "battery-shortage-predictor"

import subprocess
cmds = f"""
cd /content/battery_project
git init -q
git config user.email "you@example.com"
git config user.name "{USERNAME}"
git add .
git commit -q -m "Battery shortage prediction project"
git branch -M main
git remote remove origin 2>/dev/null
git remote add origin https://{TOKEN}@github.com/{USERNAME}/{REPO}.git
git push -u origin main
"""
print(subprocess.run(cmds, shell=True, capture_output=True, text=True))

CompletedProcess(args='\ncd /content/battery_project\ngit init -q\ngit config user.email "you@example.com"\ngit config user.name "YOUR_GITHUB_USERNAME"\ngit add .\ngit commit -q -m "Battery shortage prediction project"\ngit branch -M main\ngit remote remove origin 2>/dev/null\ngit remote add origin https://YOUR_TOKEN@github.com/YOUR_GITHUB_USERNAME/battery-shortage-predictor.git\ngit push -u origin main\n', returncode=128, stdout='', stderr="fatal: could not read Password for 'https://YOUR_TOKEN@github.com': No such device or address\n")


In [23]:
import shutil
from google.colab import files

# Compress the entire battery_project folder
shutil.make_archive('/content/battery_project_complete', 'zip', '/content/battery_project')

# Download the completed archive
files.download('/content/battery_project_complete.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>